In [5]:
import pandas as pd
import re


def clean_r_array_string(text_str):
    """Convert c("A", "B") style text into a clean comma-separated string."""
    if pd.isna(text_str):
        return ""

    text = str(text_str).strip()

    if text.startswith('c(') and text.endswith(')'):
        text = text[2:-1]

    text = re.sub(r'["\']', '', text)
    return text.strip()


def parse_iso_time(time_str):
    """Convert ISO 8601 duration (e.g., PT24H45M) to total minutes."""
    if pd.isna(time_str):
        return 0

    time_str = str(time_str)

    h_match = re.search(r'(\d+)H', time_str)
    m_match = re.search(r'(\d+)M', time_str)

    hours = int(h_match.group(1)) if h_match else 0
    minutes = int(m_match.group(1)) if m_match else 0

    return (hours * 60) + minutes


df = pd.read_csv('../data/recipes.csv')

columns_to_drop = [
    'AuthorId',
    'AuthorName',
    'ReviewCount',
    'DatePublished',
    'RecipeIngredientQuantities',
    'RecipeServings',
    'RecipeYield'
]

df = df.drop(columns=columns_to_drop, errors='ignore')

array_columns = [
    'Images',
    'Keywords',
    'RecipeIngredientParts',
    'RecipeInstructions'
]

for col in array_columns:
    if col in df.columns:
        df[col] = df[col].apply(clean_r_array_string)

time_columns = ['CookTime', 'PrepTime', 'TotalTime']

for col in time_columns:
    if col in df.columns:
        df[f'{col}Mins'] = df[col].apply(parse_iso_time)
        df = df.drop(columns=[col])

df = df.dropna(subset=['Name', 'RecipeIngredientParts', 'RecipeInstructions'])

df = df.fillna({
    'Description': '',
    'RecipeCategory': 'Uncategorized',
    'AggregatedRating': 0.0,
    'Calories': 0.0,
    'FatContent': 0.0,
    'SaturatedFatContent': 0.0,
    'CholesterolContent': 0.0,
    'SodiumContent': 0.0,
    'CarbohydrateContent': 0.0,
    'FiberContent': 0.0,
    'SugarContent': 0.0,
    'ProteinContent': 0.0
})

output_file = '../data/cleaned_recipes.csv'
df.to_csv(output_file, index=False)

df[['Name', 'RecipeCategory', 'TotalTimeMins', 'Calories']].head(3)

,Name,RecipeCategory,TotalTimeMins,Calories
0,Low-Fat Berry Blue Frozen Dessert,Frozen Desserts,1485,170.9
1,Biryani,Chicken Breast,265,1110.7
2,Best Lemonade,Beverages,35,311.1


In [7]:
df.head(50)

,RecipeId,Name,Description,Images,RecipeCategory,Keywords,RecipeIngredientParts,AggregatedRating,Calories,FatContent,...,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeInstructions,CookTimeMins,PrepTimeMins,TotalTimeMins
0,38,Low-Fat Berry Blue Frozen Dessert,Make and share this Low-Fat Berry Blue Frozen ...,https://img.sndimg.com/food/image/upload/w_555...,Frozen Desserts,"Dessert, Low Protein, Low Cholesterol, Healthy...","blueberries, granulated sugar, vanilla yogurt,...",4.5,170.9,2.5,...,8.0,29.8,37.1,3.6,30.2,3.2,"Toss 2 cups berries with sugar., Let stand for...",1440,45,1485
1,39,Biryani,Make and share this Biryani recipe from Food.com.,https://img.sndimg.com/food/image/upload/w_555...,Chicken Breast,"Chicken Thigh & Leg, Chicken, Poultry, Meat, A...","saffron, milk, hot green chili peppers, onions...",3.0,1110.7,58.8,...,372.8,368.4,84.4,9.0,20.4,63.4,Soak saffron in warm milk for 5 minutes and pu...,25,240,265
2,40,Best Lemonade,This is from one of my first Good House Keepi...,https://img.sndimg.com/food/image/upload/w_555...,Beverages,"Low Protein, Low Cholesterol, Healthy, Summer,...","sugar, lemons, rind of, lemon, zest of, fresh ...",4.5,311.1,0.2,...,0.0,1.8,81.5,0.4,77.2,0.3,"Into a 1 quart Jar with tight fitting lid, put...",5,30,35
3,41,Carina's Tofu-Vegetable Kebabs,This dish is best prepared a day in advance to...,https://img.sndimg.com/food/image/upload/w_555...,Soy/Tofu,"Beans, Vegetable, Low Cholesterol, Weeknight, ...","extra firm tofu, eggplant, zucchini, mushrooms...",4.5,536.1,24.0,...,0.0,1558.6,64.2,17.3,32.1,29.3,"Drain the tofu, carefully squeezing out excess...",20,1440,1460
4,42,Cabbage Soup,Make and share this Cabbage Soup recipe from F...,https://img.sndimg.com/food/image/upload/w_555...,Vegetable,"Low Protein, Vegan, Low Cholesterol, Healthy, ...","plain tomato juice, cabbage, onion, carrots, c...",4.5,103.6,0.4,...,0.0,959.3,25.1,4.8,17.7,4.3,"Mix everything together and bring to a boil., ...",30,20,50
5,43,Best Blackbottom Pie,Make and share this Best Blackbottom Pie recip...,character(0),Pie,"Dessert, Weeknight, Stove Top, < 4 Hours","graham cracker crumbs, sugar, butter, sugar, c...",1.0,437.9,19.3,...,94.3,267.6,58.0,1.8,42.5,7.0,"Graham Cracker Crust: In small bowl, combine g...",120,20,140
6,44,Warm Chicken A La King,I copied this one out of a friend's book so ma...,https://img.sndimg.com/food/image/upload/w_555...,Chicken,"Poultry, Meat, < 60 Mins","chicken, butter, flour, milk, celery, button m...",5.0,895.5,66.8,...,405.8,557.2,29.1,3.1,5.0,45.3,"Melt 1 1/2 ozs butter, add the flour and cook ...",3,35,38
7,45,Buttermilk Pie With Gingersnap Crumb Crust,Make and share this Buttermilk Pie With Ginger...,https://img.sndimg.com/food/image/upload/w_555...,Pie,"Dessert, Healthy, Weeknight, Oven, < 4 Hours","sugar, margarine, egg, flour, salt, buttermilk...",4.0,228.0,7.1,...,24.5,281.8,37.5,0.5,24.7,4.2,"Preheat oven to 350°F., Make pie crust, using ...",50,30,80
8,46,A Jad - Cucumber Pickle,Make and share this A Jad - Cucumber Pickle re...,character(0),Vegetable,"Thai, Asian, Free Of..., < 30 Mins","rice vinegar, haeo",5.0,4.3,0.0,...,0.0,0.7,1.1,0.2,0.2,0.1,"Slice the cucumber in four lengthwise, then sl...",0,25,25
9,47,Butter Pecan Cookies,Make and share this Butter Pecan Cookies recip...,https://img.sndimg.com/food/image/upload/w_555...,Dessert,"Cookie & Brownie, Fruit, Nuts, Weeknight, Oven...","butter, brown sugar, granulated sugar, vanilla...",4.0,69.0,5.6,...,6.3,15.0,4.5,0.6,1.6,0.8,"Preheat oven to 350 degrees., Cream butter in ...",9,55,64


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522517 entries, 0 to 522516
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   RecipeId               522517 non-null  int64  
 1   Name                   522517 non-null  object 
 2   Description            522517 non-null  object 
 3   Images                 522517 non-null  object 
 4   RecipeCategory         522517 non-null  object 
 5   Keywords               522517 non-null  object 
 6   RecipeIngredientParts  522517 non-null  object 
 7   AggregatedRating       522517 non-null  float64
 8   Calories               522517 non-null  float64
 9   FatContent             522517 non-null  float64
 10  SaturatedFatContent    522517 non-null  float64
 11  CholesterolContent     522517 non-null  float64
 12  SodiumContent          522517 non-null  float64
 13  CarbohydrateContent    522517 non-null  float64
 14  FiberContent           522517 non-nu

## Fill image

In [13]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
from sklearn.metrics.pairwise import linear_kernel

print("Filtering recipes with and without images...")

def is_empty_img(img):
    img_str = str(img).strip()
    return pd.isna(img) or img_str == "" or img_str == "character(0)" or "placeholder" in img_str

df_has_img = df[~df['Images'].apply(is_empty_img)].copy()
df_no_img = df[df['Images'].apply(is_empty_img)].copy()

print(f"Recipes with images (Donors): {len(df_has_img)}")
print(f"Recipes without images (Receivers): {len(df_no_img)}")

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

print("Computing TF-IDF matrix...")
tfidf_matrix_has = vectorizer.fit_transform(df_has_img['Name'].fillna(""))

print("Transforming receivers...")
tfidf_matrix_no = vectorizer.transform(df_no_img['Name'].fillna(""))

print("Computing similarity in batches...")

batch_size = 5000
best_indices = []

for i in tqdm(range(0, tfidf_matrix_no.shape[0], batch_size)):
    batch = tfidf_matrix_no[i:i+batch_size]

    sim = linear_kernel(batch, tfidf_matrix_has)  # faster than cosine_similarity
    batch_best = np.argmax(sim, axis=1)

    best_indices.extend(batch_best)

print("Assigning images...")

df_no_img['Images'] = df_has_img.iloc[best_indices]['Images'].values

df_final = pd.concat([df_has_img, df_no_img]).sort_values(by='RecipeId')

print("Completed. All missing images have been filled.")

df_final.to_csv('../data/cleaned_recipes_v2.csv', index=False)

Filtering recipes with and without images...
Recipes with images (Donors): 165896
Recipes without images (Receivers): 356621
Computing TF-IDF matrix...
Transforming receivers...
Computing similarity in batches...


100%|██████████| 72/72 [04:12<00:00,  3.51s/it]


Assigning images...
Completed. All missing images have been filled.


In [19]:
df_final.head(5)

,RecipeId,Name,Description,Images,RecipeCategory,Keywords,RecipeIngredientParts,AggregatedRating,Calories,FatContent,...,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeInstructions,CookTimeMins,PrepTimeMins,TotalTimeMins
0,38,Low-Fat Berry Blue Frozen Dessert,Make and share this Low-Fat Berry Blue Frozen ...,https://img.sndimg.com/food/image/upload/w_555...,Frozen Desserts,"Dessert, Low Protein, Low Cholesterol, Healthy...","blueberries, granulated sugar, vanilla yogurt,...",4.5,170.9,2.5,...,8.0,29.8,37.1,3.6,30.2,3.2,"Toss 2 cups berries with sugar., Let stand for...",1440,45,1485
1,39,Biryani,Make and share this Biryani recipe from Food.com.,https://img.sndimg.com/food/image/upload/w_555...,Chicken Breast,"Chicken Thigh & Leg, Chicken, Poultry, Meat, A...","saffron, milk, hot green chili peppers, onions...",3.0,1110.7,58.8,...,372.8,368.4,84.4,9.0,20.4,63.4,Soak saffron in warm milk for 5 minutes and pu...,25,240,265
2,40,Best Lemonade,This is from one of my first Good House Keepi...,https://img.sndimg.com/food/image/upload/w_555...,Beverages,"Low Protein, Low Cholesterol, Healthy, Summer,...","sugar, lemons, rind of, lemon, zest of, fresh ...",4.5,311.1,0.2,...,0.0,1.8,81.5,0.4,77.2,0.3,"Into a 1 quart Jar with tight fitting lid, put...",5,30,35
3,41,Carina's Tofu-Vegetable Kebabs,This dish is best prepared a day in advance to...,https://img.sndimg.com/food/image/upload/w_555...,Soy/Tofu,"Beans, Vegetable, Low Cholesterol, Weeknight, ...","extra firm tofu, eggplant, zucchini, mushrooms...",4.5,536.1,24.0,...,0.0,1558.6,64.2,17.3,32.1,29.3,"Drain the tofu, carefully squeezing out excess...",20,1440,1460
4,42,Cabbage Soup,Make and share this Cabbage Soup recipe from F...,https://img.sndimg.com/food/image/upload/w_555...,Vegetable,"Low Protein, Vegan, Low Cholesterol, Healthy, ...","plain tomato juice, cabbage, onion, carrots, c...",4.5,103.6,0.4,...,0.0,959.3,25.1,4.8,17.7,4.3,"Mix everything together and bring to a boil., ...",30,20,50


In [20]:

text_columns = ['Keywords', 'RecipeCategory', 'RecipeIngredientParts', 'Description']

for col in text_columns:
    if col in df_final.columns:
        df_final[col] = df_final[col].astype(str).str.replace(r'(?i)<null>|NA|NaN', '', regex=True)
        df_final[col] = df_final[col].str.strip(' ,')

df_final.to_csv('../data/recipes_ready_for_es.csv', index=False)

In [23]:
import pandas as pd

# file_path = "../data/cleaned_recipes.csv"
# file_path = "../data/cleaned_recipes_v2.csv"
file_path = "../data/recipes_ready_for_es.csv"

df = pd.read_csv(file_path)

def is_empty_img(img):
    if pd.isna(img):
        return True
    s = str(img).strip().lower()
    return (
        s == ""
        or s == "character(0)"
        or "placeholder" in s
        or s in {"na", "nan", "null", "none", "[]"}
    )

# Check if Images column exists
if "Images" not in df.columns:
    raise ValueError("Column `Images` not found in this file.")

total_rows = len(df)
missing_mask = df["Images"].apply(is_empty_img)
missing_rows = int(missing_mask.sum())
has_img_rows = total_rows - missing_rows
missing_pct = (missing_rows / total_rows * 100) if total_rows else 0.0

# Count total image links (if a row has multiple links separated by commas)
def count_links(cell):
    if is_empty_img(cell):
        return 0
    parts = [p.strip() for p in str(cell).split(",")]
    return sum(1 for p in parts if p)

total_image_links = int(df["Images"].apply(count_links).sum())

print(f"File: {file_path}")
print(f"Total rows: {total_rows:,}")
print(f"Rows with images: {has_img_rows:,}")
print(f"Rows without images: {missing_rows:,} ({missing_pct:.2f}%)")
print(f"Total image links: {total_image_links:,}")

if missing_rows == 0:
    print("Result: All rows have images ✅")
else:
    print("Result: Some rows are missing images ❌")
    print("\nSample rows with missing images:")
    cols = [c for c in ["RecipeId", "Name", "Images"] if c in df.columns]
    print(df.loc[missing_mask, cols].head(10).to_string(index=False))

File: ../data/recipes_ready_for_es.csv
Total rows: 522,517
Rows with images: 522,517
Rows without images: 0 (0.00%)
Total image links: 6,924,085
Result: All rows have images ✅
